## KuaiRand 数据理解与质量检查

#### 1. 数据表结构
#### 2. 数据粒度与关联关系
#### 3. 主键唯一性
#### 4. 表间覆盖关系
#### 5. 重复数据检查
#### 6. 缺失值与字段合法性
#### 7. 数据质量结论


In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data/raw/KuaiRand-Pure/data"

print("当前目录：", Path.cwd())
print("项目目录：", PROJECT_ROOT)
print("数据目录存在：", DATA_DIR.exists())

当前目录： /Users/nora/Documents/GitHub/kuairand-product-analytics/notebooks
项目目录： /Users/nora/Documents/GitHub/kuairand-product-analytics
数据目录存在： True


In [3]:
files = sorted(DATA_DIR.glob("*.csv"))

for file in files:
    sample = pd.read_csv(file, nrows=5)

    print("=" * 60)
    print("表名：", file.name)
    print("字段数量：", sample.shape[1])
    print("字段：", sample.columns.tolist())

表名： log_random_4_22_to_5_08_pure.csv
字段数量： 19
字段： ['user_id', 'video_id', 'date', 'hourmin', 'time_ms', 'is_click', 'is_like', 'is_follow', 'is_comment', 'is_forward', 'is_hate', 'long_view', 'play_time_ms', 'duration_ms', 'profile_stay_time', 'comment_stay_time', 'is_profile_enter', 'is_rand', 'tab']
表名： log_standard_4_08_to_4_21_pure.csv
字段数量： 19
字段： ['user_id', 'video_id', 'date', 'hourmin', 'time_ms', 'is_click', 'is_like', 'is_follow', 'is_comment', 'is_forward', 'is_hate', 'long_view', 'play_time_ms', 'duration_ms', 'profile_stay_time', 'comment_stay_time', 'is_profile_enter', 'is_rand', 'tab']
表名： log_standard_4_22_to_5_08_pure.csv
字段数量： 19
字段： ['user_id', 'video_id', 'date', 'hourmin', 'time_ms', 'is_click', 'is_like', 'is_follow', 'is_comment', 'is_forward', 'is_hate', 'long_view', 'play_time_ms', 'duration_ms', 'profile_stay_time', 'comment_stay_time', 'is_profile_enter', 'is_rand', 'tab']
表名： user_features_pure.csv
字段数量： 31
字段： ['user_id', 'user_active_degree', 'is_lowactive

## 数据表结构总结

KuaiRand-Pure 包含三类数据表：

1. **行为日志表**
   - 三张日志表的一行代表一次用户与视频的曝光及交互。
   - 包含用户、视频、时间以及点击、点赞、关注、评论和长播等反馈。
   - 随机曝光日志与正常推荐日志的曝光机制不同。

2. **用户特征表**
   - 一行代表一名用户。
   - 通过 `user_id` 与行为日志连接。

3. **视频特征表**
   - 一行代表一个视频。
   - 基础特征表记录视频属性，统计特征表记录历史表现。
   - 通过 `video_id` 与行为日志连接。

In [4]:
#1. 读取三张特征表
user_path = DATA_DIR / "user_features_pure.csv"
video_basic_path = DATA_DIR / "video_features_basic_pure.csv"
video_stat_path = DATA_DIR / "video_features_statistic_pure.csv"

users = pd.read_csv(user_path)
video_basic = pd.read_csv(video_basic_path)
video_stat = pd.read_csv(video_stat_path)

In [5]:
# 2. 检查行数和唯一 ID 数量
print("用户表")
print("总行数：", len(users))
print("唯一用户数：", users["user_id"].nunique())
print("重复 user_id 数：", users["user_id"].duplicated().sum())

print("\n视频基础表")
print("总行数：", len(video_basic))
print("唯一视频数：", video_basic["video_id"].nunique())
print("重复 video_id 数：", video_basic["video_id"].duplicated().sum())

print("\n视频统计表")
print("总行数：", len(video_stat))
print("唯一视频数：", video_stat["video_id"].nunique())
print("重复 video_id 数：", video_stat["video_id"].duplicated().sum())

用户表
总行数： 27285
唯一用户数： 27285
重复 user_id 数： 0

视频基础表
总行数： 7583
唯一视频数： 7583
重复 video_id 数： 0

视频统计表
总行数： 7583
唯一视频数： 7583
重复 video_id 数： 0


In [6]:
#3. 检查两张视频表是否覆盖相同视频
basic_ids = set(video_basic["video_id"])
stat_ids = set(video_stat["video_id"])

print("基础表有、统计表没有：", len(basic_ids - stat_ids))
print("统计表有、基础表没有：", len(stat_ids - basic_ids))
print("两张表共有的视频：", len(basic_ids & stat_ids))

基础表有、统计表没有： 0
统计表有、基础表没有： 0
两张表共有的视频： 7583


## 主键检查结论

- 用户表中 `user_id` 唯一，可以作为主键。
- 两张视频表中 `video_id` 唯一，可以作为主键。
- 两张视频表覆盖相同的视频，可通过 `video_id` 一对一合并。
- 如果右表连接键不唯一，合并会导致行为记录被复制，从而造成指标重复计算。

In [7]:
log_files = [
    "log_random_4_22_to_5_08_pure.csv",
    "log_standard_4_08_to_4_21_pure.csv",
    "log_standard_4_22_to_5_08_pure.csv"
]

for filename in log_files:
    log = pd.read_csv(DATA_DIR / filename)

    print("=" * 60)
    print("表名：", filename)
    print("记录数：", len(log))
    print("用户数：", log["user_id"].nunique())
    print("视频数：", log["video_id"].nunique())
    print("日期范围：", log["date"].min(), "-", log["date"].max())
    print("is_rand 分布：")
    print(log["is_rand"].value_counts(dropna=False))

表名： log_random_4_22_to_5_08_pure.csv
记录数： 1186059
用户数： 27285
视频数： 7583
日期范围： 20220422 - 20220508
is_rand 分布：
is_rand
1    1186059
Name: count, dtype: int64
表名： log_standard_4_08_to_4_21_pure.csv
记录数： 1141112
用户数： 26210
视频数： 7538
日期范围： 20220409 - 20220421
is_rand 分布：
is_rand
0    1141112
Name: count, dtype: int64
表名： log_standard_4_22_to_5_08_pure.csv
记录数： 295497
用户数： 25877
视频数： 6618
日期范围： 20220422 - 20220508
is_rand 分布：
is_rand
0    295497
Name: count, dtype: int64


In [8]:
log_random = pd.read_csv(
    DATA_DIR / "log_random_4_22_to_5_08_pure.csv"
)

print(
    "重复用户-视频组合数量：",
    log_random.duplicated(
        subset=["user_id", "video_id"]
    ).sum()
)

print(
    "完全重复记录数量：",
    log_random.duplicated().sum()
)

重复用户-视频组合数量： 53
完全重复记录数量： 10


In [9]:
exact_duplicates = log_random[
    log_random.duplicated(keep=False)
].sort_values(["user_id", "video_id", "time_ms"])

exact_duplicates

,user_id,video_id,date,hourmin,time_ms,is_click,is_like,is_follow,is_comment,is_forward,is_hate,long_view,play_time_ms,duration_ms,profile_stay_time,comment_stay_time,is_profile_enter,is_rand,tab
40982,940,4121,20220502,2200,1651499805466,0,0,0,0,0,0,0,0,306916,0,0,0,1,11
40983,940,4121,20220502,2200,1651499805466,0,0,0,0,0,0,0,0,306916,0,0,0,1,11
82430,1944,5121,20220501,2200,1651413628750,0,0,0,0,0,0,0,0,97450,0,0,0,1,11
82431,1944,5121,20220501,2200,1651413628750,0,0,0,0,0,0,0,0,97450,0,0,0,1,11
288739,6778,4479,20220503,2000,1651579839722,0,0,0,0,0,0,0,0,131300,0,0,0,1,11
288740,6778,4479,20220503,2000,1651579839722,0,0,0,0,0,0,0,0,131300,0,0,0,1,11
307928,7270,708,20220429,2200,1651241594702,0,0,0,0,0,0,0,0,165080,0,0,0,1,11
307929,7270,708,20220429,2200,1651241594702,0,0,0,0,0,0,0,0,165080,0,0,0,1,11
431054,9966,4730,20220423,2000,1650715516025,0,0,0,0,0,0,0,0,60816,0,0,0,1,11
431055,9966,4730,20220423,2000,1650715516025,0,0,0,0,0,0,0,0,60816,0,0,0,1,11


In [10]:
log_random_clean = log_random.drop_duplicates().copy()

print("处理前记录数：", len(log_random))
print("处理后记录数：", len(log_random_clean))
print("删除记录数：", len(log_random) - len(log_random_clean))
print("处理后完全重复数：", log_random_clean.duplicated().sum())

处理前记录数： 1186059
处理后记录数： 1186049
删除记录数： 10
处理后完全重复数： 0


In [11]:
log_standard_before = pd.read_csv(
    DATA_DIR / "log_standard_4_08_to_4_21_pure.csv"
)

log_standard_after = pd.read_csv(
    DATA_DIR / "log_standard_4_22_to_5_08_pure.csv"
)

for name, log in {
    "前期正常推荐日志": log_standard_before,
    "同期正常推荐日志": log_standard_after
}.items():
    duplicate_count = log.duplicated().sum()

    print(name)
    print("处理前记录数：", len(log))
    print("完全重复数：", duplicate_count)
    print("-" * 40)

前期正常推荐日志
处理前记录数： 1141112
完全重复数： 15609
----------------------------------------
同期正常推荐日志
处理前记录数： 295497
完全重复数： 6378
----------------------------------------


In [12]:
log_standard_before_clean = (
    log_standard_before.drop_duplicates().copy()
)

log_standard_after_clean = (
    log_standard_after.drop_duplicates().copy()
)

print(
    "前期日志处理后：",
    len(log_standard_before_clean)
)

print(
    "同期日志处理后：",
    len(log_standard_after_clean)
)

前期日志处理后： 1125503
同期日志处理后： 289119


两张正常推荐日志分别发现15,609条和6,378条多余的完全重复记录，重复比例约为1.37%和2.16%。完全重复会导致曝光及行为指标重复计算，因此每组保留一条，其余副本从分析数据中删除。



In [13]:
clean_logs = {
    "随机曝光日志": log_random_clean,
    "前期正常推荐日志": log_standard_before_clean,
    "同期正常推荐日志": log_standard_after_clean
}

binary_columns = [
    "is_click",
    "is_like",
    "is_follow",
    "is_comment",
    "is_forward",
    "is_hate",
    "long_view",
    "is_profile_enter",
    "is_rand"
]

for log_name, log in clean_logs.items():
    print("=" * 60)
    print(log_name)

    for column in binary_columns:
        values = sorted(log[column].dropna().unique())
        print(f"{column}: {values}")

随机曝光日志
is_click: [np.int64(0), np.int64(1)]
is_like: [np.int64(0), np.int64(1)]
is_follow: [np.int64(0), np.int64(1)]
is_comment: [np.int64(0), np.int64(1)]
is_forward: [np.int64(0), np.int64(1)]
is_hate: [np.int64(0), np.int64(1)]
long_view: [np.int64(0), np.int64(1)]
is_profile_enter: [np.int64(0), np.int64(1)]
is_rand: [np.int64(1)]
前期正常推荐日志
is_click: [np.int64(0), np.int64(1)]
is_like: [np.int64(0), np.int64(1)]
is_follow: [np.int64(0), np.int64(1)]
is_comment: [np.int64(0), np.int64(1)]
is_forward: [np.int64(0), np.int64(1)]
is_hate: [np.int64(0), np.int64(1)]
long_view: [np.int64(0), np.int64(1)]
is_profile_enter: [np.int64(0), np.int64(1)]
is_rand: [np.int64(0)]
同期正常推荐日志
is_click: [np.int64(0), np.int64(1)]
is_like: [np.int64(0), np.int64(1)]
is_follow: [np.int64(0), np.int64(1)]
is_comment: [np.int64(0), np.int64(1)]
is_forward: [np.int64(0), np.int64(1)]
is_hate: [np.int64(0), np.int64(1)]
long_view: [np.int64(0), np.int64(1)]
is_profile_enter: [np.int64(0), np.int64(1)]
is_ra

In [14]:
user_ids = set(users["user_id"])
video_ids = set(video_basic["video_id"])

for name, log in clean_logs.items():
    unknown_users = set(log["user_id"]) - user_ids
    unknown_videos = set(log["video_id"]) - video_ids

    print("=" * 60)
    print(name)
    print("无法匹配的用户数：", len(unknown_users))
    print("无法匹配的视频数：", len(unknown_videos))
    
    unmatched_user_rows = ~log["user_id"].isin(user_ids)
    unmatched_video_rows = ~log["video_id"].isin(video_ids)

    print("用户无法匹配的记录数：", unmatched_user_rows.sum())
    print("视频无法匹配的记录数：", unmatched_video_rows.sum())

随机曝光日志
无法匹配的用户数： 0
无法匹配的视频数： 0
用户无法匹配的记录数： 0
视频无法匹配的记录数： 0
前期正常推荐日志
无法匹配的用户数： 0
无法匹配的视频数： 0
用户无法匹配的记录数： 0
视频无法匹配的记录数： 0
同期正常推荐日志
无法匹配的用户数： 0
无法匹配的视频数： 0
用户无法匹配的记录数： 0
视频无法匹配的记录数： 0


In [15]:
print("关联前：", len(log_random_clean))

merged_check = (
    log_random_clean
    .merge(
        users[["user_id"]],
        on="user_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        video_basic[["video_id"]],
        on="video_id",
        how="left",
        validate="many_to_one"
    )
)

print("关联后：", len(merged_check))

关联前： 1186049
关联后： 1186049


In [16]:
for log_name, log in clean_logs.items():
    missing = log.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    print("=" * 60)
    print(log_name)

    if missing.empty:
        print("没有缺失值")
    else:
        print(missing)

随机曝光日志
没有缺失值
前期正常推荐日志
没有缺失值
同期正常推荐日志
没有缺失值


### 字段完整性与合法性结论

- 三张日志表均未发现缺失值。
- 点击、点赞、关注、评论、转发、负反馈、长播等二分类字段仅包含合法的 0/1。
- 随机日志的 `is_rand` 全为 1，正常推荐日志全为 0。
- 暂时不需要进行缺失值填补或行为字段修正。

In [17]:
duration_columns = [
    "play_time_ms",
    "duration_ms",
    "profile_stay_time",
    "comment_stay_time"
]

for log_name, log in clean_logs.items():
    print("=" * 60)
    print(log_name)

    summary = log[duration_columns].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    ).T

    display(summary)

随机曝光日志


,count,mean,std,min,50%,90%,95%,99%,max
play_time_ms,1186049.0,6935.569776,18791.625894,0.0,2091.0,13003.0,27887.0,91337.12,1023809.0
duration_ms,1186049.0,104438.633811,104987.390149,0.0,76833.0,232966.0,298450.0,507666.00,1177720.0
profile_stay_time,1186049.0,0.417857,135.496660,0.0,0.0,0.0,0.0,0.00,89503.0
comment_stay_time,1186049.0,77.354592,1505.161319,0.0,0.0,0.0,0.0,0.00,300000.0


前期正常推荐日志


,count,mean,std,min,50%,90%,95%,99%,max
play_time_ms,1125503.0,23490.623735,43789.905595,0.0,5070.0,70494.0,107792.7,214088.98,964774.0
duration_ms,1125503.0,98340.460314,95504.632816,0.0,71033.0,235766.0,286166.0,420483.00,1177720.0
profile_stay_time,1125503.0,3.351874,762.872676,0.0,0.0,0.0,0.0,0.00,300000.0
comment_stay_time,1125503.0,559.044213,4106.716559,0.0,0.0,0.0,1851.9,14768.96,300000.0


同期正常推荐日志


,count,mean,std,min,50%,90%,95%,99%,max
play_time_ms,289119.0,21790.113227,42132.410389,0.0,4831.0,63380.0,101060.2,208297.92,898147.0
duration_ms,289119.0,104698.828154,106099.043440,0.0,70566.0,252900.0,306500.0,445266.00,1177720.0
profile_stay_time,289119.0,2.309084,630.643692,0.0,0.0,0.0,0.0,0.00,300000.0
comment_stay_time,289119.0,467.228415,3729.149951,0.0,0.0,0.0,0.0,13068.66,300000.0


In [18]:
for log_name, log in clean_logs.items():
    print("=" * 60)
    print(log_name)

    for column in duration_columns:
        print(
            column,
            "负数数量：", (log[column] < 0).sum(),
            "零值比例：", f"{(log[column] == 0).mean():.2%}"
        )

    replay_count = (log["play_time_ms"] > log["duration_ms"]).sum()

    print(
        "播放时长超过视频时长：",
        replay_count,
        f"占比：{replay_count / len(log):.2%}"
    )

随机曝光日志
play_time_ms 负数数量： 0 零值比例： 0.67%
duration_ms 负数数量： 0 零值比例： 3.11%
profile_stay_time 负数数量： 0 零值比例： 100.00%
comment_stay_time 负数数量： 0 零值比例： 99.01%
播放时长超过视频时长： 74841 占比：6.31%
前期正常推荐日志
play_time_ms 负数数量： 0 零值比例： 14.06%
duration_ms 负数数量： 0 零值比例： 2.14%
profile_stay_time 负数数量： 0 零值比例： 99.99%
comment_stay_time 负数数量： 0 零值比例： 94.51%
播放时长超过视频时长： 193037 占比：17.15%
同期正常推荐日志
play_time_ms 负数数量： 0 零值比例： 10.54%
duration_ms 负数数量： 0 零值比例： 1.66%
profile_stay_time 负数数量： 0 零值比例： 99.99%
comment_stay_time 负数数量： 0 零值比例： 95.50%
播放时长超过视频时长： 44677 占比：15.45%


- 各时长字段均未发现负数，不需要针对负值进行删除或修正。

In [19]:
for name, log in clean_logs.items():
    zero_count = (log["duration_ms"] == 0).sum()
    print(name, "视频时长为0：", zero_count)

随机曝光日志 视频时长为0： 36920
前期正常推荐日志 视频时长为0： 24062
同期正常推荐日志 视频时长为0： 4798


In [20]:
for name, log in clean_logs.items():
    zero_mask = log["duration_ms"] == 0

    print(
        name,
        f"零时长记录：{zero_mask.sum():,}",
        f"占比：{zero_mask.mean():.2%}",
        f"涉及视频：{log.loc[zero_mask, 'video_id'].nunique():,}"
    )

随机曝光日志 零时长记录：36,920 占比：3.11% 涉及视频：239
前期正常推荐日志 零时长记录：24,062 占比：2.14% 涉及视频：237
同期正常推荐日志 零时长记录：4,798 占比：1.66% 涉及视频：190


In [21]:
zero_duration = (
    log_random_clean.loc[
        log_random_clean["duration_ms"] == 0,
        ["video_id", "duration_ms"]
    ]
    .merge(
        video_basic[["video_id", "video_duration"]],
        on="video_id",
        how="left"
    )
)

zero_duration["video_duration"].describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: video_duration, dtype: float64

In [22]:
print("成功匹配的行数：", zero_duration["video_id"].notna().sum())
print("video_duration 缺失数：", zero_duration["video_duration"].isna().sum())
print("video_duration 非缺失数：", zero_duration["video_duration"].notna().sum())

成功匹配的行数： 36920
video_duration 缺失数： 36920
video_duration 非缺失数： 0


In [23]:
valid_duration_map = (
    log_random_clean.loc[
        log_random_clean["duration_ms"] > 0
    ]
    .groupby("video_id")["duration_ms"]
    .median()
)

zero_video_ids = set(
    log_random_clean.loc[
        log_random_clean["duration_ms"] == 0,
        "video_id"
    ]
)

recoverable_ids = zero_video_ids & set(valid_duration_map.index)

print("零时长视频数：", len(zero_video_ids))
print("可从其他记录恢复的视频数：", len(recoverable_ids))

零时长视频数： 239
可从其他记录恢复的视频数： 0


In [24]:
all_standard_logs = pd.concat(
    [
        log_standard_before_clean,
        log_standard_after_clean
    ],
    ignore_index=True
)

standard_duration_map = (
    all_standard_logs.loc[
        all_standard_logs["duration_ms"] > 0
    ]
    .groupby("video_id")["duration_ms"]
    .median()
)

recoverable_from_standard = (
    zero_video_ids & set(standard_duration_map.index)
)

print("可从正常推荐日志恢复的视频数：",
      len(recoverable_from_standard))

可从正常推荐日志恢复的视频数： 0


In [25]:
log_random_clean["duration_missing"] = (
    log_random_clean["duration_ms"] == 0
).astype(int)

log_random_clean["play_ratio"] = pd.NA

valid_duration = log_random_clean["duration_ms"] > 0

log_random_clean.loc[valid_duration, "play_ratio"] = (
    log_random_clean.loc[valid_duration, "play_time_ms"]
    / log_random_clean.loc[valid_duration, "duration_ms"]
)

### 零视频时长处理结论

部分日志记录的 `duration_ms` 为 0，且无法从视频基础表或其他日志恢复。

这些记录的曝光、点击和互动信息仍然有效，因此不删除整行，也不强行填补时长。后续处理规则如下：

- 曝光、点击、点赞等分析继续保留；
- 计算播放比例、完播率时排除；
- 新增 `duration_missing` 标记；
- 原始字段保持不变。

In [26]:
time_check = log_random_clean[
    ["date", "hourmin", "time_ms"]
].copy()

time_check["datetime"] = pd.to_datetime(
    time_check["time_ms"],
    unit="ms"
)

time_check.head()

,date,hourmin,time_ms,datetime
0,20220430,1800,1651314030792,2022-04-30 10:20:30.792
1,20220502,1200,1651466607423,2022-05-02 04:43:27.423
2,20220502,1700,1651481542743,2022-05-02 08:52:22.743
3,20220502,1800,1651488577163,2022-05-02 10:49:37.163
4,20220503,800,1651535940163,2022-05-02 23:59:00.163


In [28]:
time_check["datetime_cn"] = (
    pd.to_datetime(
        time_check["time_ms"],
        unit="ms",
        utc=True
    )
    .dt.tz_convert("Asia/Shanghai")
)

time_check["date_from_timestamp_cn"] = (
    time_check["datetime_cn"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

date_mismatch = (
    time_check["date"]
    != time_check["date_from_timestamp_cn"]
)

print("按中国时区转换后不一致数：", date_mismatch.sum())
print("不一致比例：", f"{date_mismatch.mean():.2%}")

按中国时区转换后不一致数： 8626
不一致比例： 0.73%


In [30]:
time_check["date_parsed"] = pd.to_datetime(
    time_check["date"].astype(str),
    format="%Y%m%d"
)

time_check["timestamp_date"] = (
    time_check["datetime_cn"]
    .dt.tz_localize(None)
    .dt.normalize()
)

time_check["date_diff_days"] = (
    time_check["timestamp_date"]
    - time_check["date_parsed"]
).dt.days

time_check.loc[
    date_mismatch,
    "date_diff_days"
].value_counts()

date_diff_days
-1    8626
Name: count, dtype: int64

In [31]:
time_check.loc[
    date_mismatch,
    [
        "date",
        "hourmin",
        "time_ms",
        "datetime_cn",
        "date_from_timestamp_cn",
        "date_diff_days"
    ]
].head(20)

,date,hourmin,time_ms,datetime_cn,date_from_timestamp_cn,date_diff_days
140,20220502,0,1651420256116,2022-05-01 23:50:56.116000+08:00,20220501,-1
219,20220430,0,1651244944324,2022-04-29 23:09:04.324000+08:00,20220429,-1
220,20220430,0,1651244964517,2022-04-29 23:09:24.517000+08:00,20220429,-1
221,20220430,0,1651245119440,2022-04-29 23:11:59.440000+08:00,20220429,-1
222,20220430,0,1651245473192,2022-04-29 23:17:53.192000+08:00,20220429,-1
223,20220430,0,1651245781191,2022-04-29 23:23:01.191000+08:00,20220429,-1
224,20220430,0,1651245782652,2022-04-29 23:23:02.652000+08:00,20220429,-1
225,20220430,0,1651245814746,2022-04-29 23:23:34.746000+08:00,20220429,-1
226,20220430,0,1651245850740,2022-04-29 23:24:10.740000+08:00,20220429,-1
227,20220430,0,1651245858201,2022-04-29 23:24:18.201000+08:00,20220429,-1


In [29]:
time_check["hourmin_from_timestamp"] = (
    time_check["datetime_cn"].dt.hour * 100
    + time_check["datetime_cn"].dt.minute
)

hourmin_mismatch = (
    time_check["hourmin"]
    != time_check["hourmin_from_timestamp"]
)

print("hourmin 不一致数：", hourmin_mismatch.sum())
print("hourmin 不一致比例：", f"{hourmin_mismatch.mean():.2%}")

hourmin 不一致数： 1166628
hourmin 不一致比例： 98.36%
